# `qiu_2026` — interactive training walkthrough

A step-by-step, human-driven version of what `uv run openretina train --config-name qiu_2026_core_readout` does under the hood. You will:

1. **Compose the Hydra config** (the same YAMLs the CLI uses), the notebook way.
2. **Load and inspect the data** — the 3-channel video+behavior movie, the responses, the pupil trace — with shape checks and plots.
3. **Build `data_info` + the dataloaders** and look at one `QiuDataPoint` batch.
4. **Build the model** (core + Gaussian readout + shifter) and run a forward pass, with and without the shifter.
5. **Run a short training loop** to confirm the loss is finite and decreasing.

Each stage mirrors a specific part of `openretina/cli/train.py`, cited inline so you can compare.

---
### ⚠️ Read before running

* **RAM.** The full 10-session run needs a big box (see `qiu_2026_integration_plan.md`). This notebook **defaults to the 2 smallest sessions and a tiny batch** so it can build and inspect everything on a laptop. Stage 5's *training* cell is a **smoke test** (a couple of batches on CPU) — it will be slow; the real run belongs on the cluster / a GPU box.
* **Data location** is machine-specific — see `qiu_2026_integration_plan_dataset_location.md`. Set `DATA_DIR` / `CACHE_DIR` in the config cell below to match your machine.
* Reference material (architecture, session/neuron tables) lives in `qiu_2026_reference.md`; the settled design decisions in `qiu_2026_decisions.md`.

## 0. Configuration — **edit this cell**

Everything you might want to change lives here. Defaults target the local Mac laptop with the two smallest sessions.

In [ ]:
import os
from pathlib import Path
import openretina

# Repo layout (robust to wherever the notebook is launched from).
REPO_ROOT = Path(openretina.__file__).resolve().parents[1]
CONFIG_DIR = REPO_ROOT / "configs"

# --- Data pointers: set these per qiu_2026_integration_plan_dataset_location.md ------------------
# Cluster: leave DATA_DIR at the HF URL and CACHE_DIR at ~/openretina_cache.
# Laptop : local extracted folder (all 10 sessions are present here, ~131 GB).
DATA_DIR = "/Users/lhoefling/data/franke_lab/qiu_2026"
CACHE_DIR = "/Users/lhoefling/data"

# Where run artifacts (if any) go. Kept out of the repo tree.
OUTPUT_DIR = str(Path.home() / "openretina_notebook_runs")

# --- Which sessions to load. None => all 10 (needs a big box). ------------------------------------
# The two smallest sessions (~2.8 GB combined) — safe to load & inspect on a laptop.
SESSIONS = [
    "dynamic28188-16-5-Fluorescence-7b721b-v4a",
    "dynamic28188-16-3-Fluorescence-7b721b-v4a",
]

# --- Dataloader / training knobs ------------------------------------------------------------------
BATCH_SIZE = 4          # 32 is the config default; 4 is what fit in RAM on the 34 GB laptop
TRAIN_CHUNK_SIZE = 50   # frames per training chunk (< clip_length=300 keeps shift augmentation)
ACCELERATOR = "cpu"     # "cpu" is safest here; the debug trainer also forces CPU

# Stage 5 smoke-training size (keep tiny on CPU — each batch is seconds-to-minutes).
SMOKE_TRAIN_BATCHES = 3
SMOKE_VAL_BATCHES = 1

os.environ["OPENRETINA_CACHE_DIRECTORY"] = CACHE_DIR

# Quick sanity: which session folders are actually on disk?
data_path = Path(DATA_DIR)
if data_path.exists():
    found = sorted(p.name for p in data_path.glob("dynamic*") if p.is_dir())
    print(f"{len(found)} session folder(s) under {DATA_DIR}:")
    for name in found:
        print("   ", name)
else:
    print(f"{DATA_DIR} does not exist locally -> loaders will try to download from the HF URL.")
print("\nConfig dir:", CONFIG_DIR)

## 1. Compose the Hydra config

The CLI uses the `@hydra.main` decorator (`train.py:21`). In a notebook we use Hydra's **Compose API** instead: point it at the repo's `configs/` folder and compose the top-level `qiu_2026_core_readout` config, passing the same kind of overrides you would give on the command line.

We override the `paths.*` (to avoid the `${hydra:runtime.output_dir}` resolver, which only exists under `@hydra.main`) and `dataloader.batch_size`. The **session filter** is injected straight onto the loader configs afterwards with `open_dict` — cleaner than fighting Hydra's list-quoting for keys full of hyphens.

In [ ]:
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from omegaconf import OmegaConf, open_dict

overrides = [
    f"paths.data_dir={DATA_DIR}",
    f"paths.cache_dir={CACHE_DIR}",
    f"paths.output_dir={OUTPUT_DIR}",
    f"paths.log_dir={OUTPUT_DIR}",
    f"dataloader.batch_size={BATCH_SIZE}",
    f"dataloader.train_chunk_size={TRAIN_CHUNK_SIZE}",
]

GlobalHydra.instance().clear()  # so this cell is safe to re-run
with initialize_config_dir(config_dir=str(CONFIG_DIR), version_base="1.3"):
    cfg = compose(config_name="qiu_2026_core_readout", overrides=overrides)

# Inject the session filter onto each per-stream loader config (adds a new key -> open_dict).
if SESSIONS is not None:
    with open_dict(cfg):
        cfg.data_io.stimuli.sessions = list(SESSIONS)
        cfg.data_io.responses.sessions = list(SESSIONS)
        cfg.data_io.pupil.sessions = list(SESSIONS)

print("exp_name           :", cfg.exp_name)
print("paths.data_dir     :", cfg.paths.data_dir)
print("dataloader.batch   :", cfg.dataloader.batch_size)
print("model.in_shape     :", OmegaConf.to_container(cfg.model.in_shape))
print("sessions requested :", SESSIONS if SESSIONS is not None else "ALL (10)")

Peek at the composed sub-configs. Note `channels`, `in_shape` and `n_neurons_dict` are still `???` — they get filled in from the data in stages 3–4, exactly as `train.py` does it.

In [ ]:
print("===== data_io =====")
print(OmegaConf.to_yaml(cfg.data_io))
print("===== dataloader =====")
print(OmegaConf.to_yaml(cfg.dataloader))
print("===== model (core / readout / shifter targets) =====")
print("core   :", cfg.model.core._target_)
print("readout:", cfg.model.readout._target_, "| grid_mean_predictor =", cfg.model.readout.grid_mean_predictor)
print("shifter:", cfg.model.shifter._target_)

## 2. Load & inspect the data

`train.py:48-49,63` calls the three loaders via `hydra.utils.call`. Each returns a `{session_key: ...}` dict. This reads the per-trial `.npy` arrays into memory, so it is the first heavy step — expect it to take a little while (and a few GB of RAM) even for 2 sessions.

In [ ]:
import time
import hydra

t0 = time.time()
movies_dict = hydra.utils.call(cfg.data_io.stimuli)      # {session: MoviesTrainTestSplit}
responses_dict = hydra.utils.call(cfg.data_io.responses) # {session: ResponsesTrainTestSplit}
pupil_dict = hydra.utils.call(cfg.data_io.pupil)         # {session: {"train":..., "test_dict":...}}
print(f"loaded {len(movies_dict)} session(s) in {time.time() - t0:.1f}s")
print("session keys:", list(movies_dict))

Shapes per session, plus the container consistency check `train.py:51-53` runs.

In [ ]:
import numpy as np

for s in movies_dict:
    mv = movies_dict[s]           # .train (C, T, H, W); .test_dict {hash: (C, T, H, W)}
    rp = responses_dict[s]        # .train (N, T);        .test_dict / .test_by_trial_dict
    pp = pupil_dict[s]            # {"train": (2, T), "test_dict": {hash: (2, T)}}
    print(s)
    print(f"   movie train  {tuple(mv.train.shape)}  (C, T, H, W)  |  {len(mv.test_dict)} test conditions")
    print(f"   responses    {tuple(rp.train.shape)}  (N, T)")
    print(f"   pupil train  {tuple(pp['train'].shape)}  (2, T)")
    assert mv.train.shape[0] == 3, "expected 3 input channels (1 video + 2 behavior)"
    assert mv.train.shape[1] == rp.train.shape[1] == pp["train"].shape[1], "streams must be frame-aligned"

# Container check: movie/response test_dict keys match, shapes line up, stim_id matches.
for s, rp in responses_dict.items():
    rp.check_matching_stimulus(movies_dict[s])
print("\ncheck_matching_stimulus: OK for all sessions")

Two things worth verifying directly: **per-session video normalization** (decision D6 — the video channel should be roughly zero-mean/unit-std over train) and the **test structure** (decision D3 — ~6 natural-clip conditions, each with 15–20 repeats).

In [ ]:
s = list(movies_dict)[0]
video_ch = movies_dict[s].train[0]  # (T, H, W)
print(f"{s}")
print(f"   video channel: mean={video_ch.mean():+.3f}  std={video_ch.std():.3f}  (expect ~0 / ~1)")

# Test conditions and their repeat counts (from the per-trial test dict).
tbt = responses_dict[s].test_by_trial_dict  # {hash: (repeats, N, T)}
print(f"   {len(tbt)} test condition(s):")
for h, arr in tbt.items():
    print(f"      {h}: {arr.shape[0]} repeats, shape {tuple(arr.shape)}")

A quick visual: one video frame, the two behavior channels, a few response traces, and the pupil trace.

In [ ]:
import matplotlib.pyplot as plt

s = list(movies_dict)[0]
mv = movies_dict[s].train      # (3, T, H, W)
rp = responses_dict[s].train   # (N, T)
pp = pupil_dict[s]["train"]     # (2, T)
T_show = min(600, mv.shape[1])

fig, ax = plt.subplots(2, 2, figsize=(12, 7))
ax[0, 0].imshow(mv[0, 0], cmap="gray"); ax[0, 0].set_title(f"{s}\nvideo channel, frame 0"); ax[0, 0].axis("off")
ax[0, 1].plot(mv[1, :T_show, 0, 0], label="behavior ch A (pupil size)")
ax[0, 1].plot(mv[2, :T_show, 0, 0], label="behavior ch B (locomotion)")
ax[0, 1].set_title("behavior channels (normalized)"); ax[0, 1].set_xlabel("frame"); ax[0, 1].legend()
for n in range(min(4, rp.shape[0])):
    ax[1, 0].plot(rp[n, :T_show] + n * 1.0, lw=0.8)
ax[1, 0].set_title("4 response traces (offset)"); ax[1, 0].set_xlabel("frame")
ax[1, 1].plot(pp[0, :T_show], label="pupil x"); ax[1, 1].plot(pp[1, :T_show], label="pupil y")
ax[1, 1].set_title("pupil center (normalized -> shifter input)"); ax[1, 1].set_xlabel("frame"); ax[1, 1].legend()
plt.tight_layout(); plt.show()

## 3. `data_info`, dataloaders, and one batch

`compute_data_info` (`train.py:67`) derives `n_neurons_dict`, the input shape, mean activity, and normalization stats from the loaded data, merging in the `data_info` block from the config. This is what fills the model's `???` placeholders.

In [ ]:
from openretina.data_io.base import compute_data_info

data_info = compute_data_info(
    responses_dict, movies_dict, partial_data_info=cfg.data_io.get("data_info")
)
print("data_info keys:", list(data_info))
print("input_shape   :", data_info["input_shape"])
print("n_neurons_dict:", dict(data_info["n_neurons_dict"]))

Build the dataloaders. `train.py:55-65` assembles a kwargs dict (adding `pupil_dictionary` because this dataset has a `pupil` block) and instantiates `cfg.dataloader`. The result is `dict[split][session] -> DataLoader`, where the splits are `train`, `validation`, and one entry per test condition.

In [ ]:
dataloaders = hydra.utils.instantiate(
    cfg.dataloader,
    neuron_data_dictionary=responses_dict,
    movies_dictionary=movies_dict,
    pupil_dictionary=pupil_dict,
)
print("splits:", list(dataloaders)[:4], "...", f"({len(dataloaders)} total incl. per-condition test splits)")
print("train sessions:", list(dataloaders["train"]))

Grab one batch. A qiu batch is a **`QiuDataPoint`** — the qiu-local 3-field namedtuple (decision D1) carrying `pupil_center` alongside `inputs`/`targets`, so existing 2-field datasets are untouched.

In [ ]:
sess = list(dataloaders["train"])[0]
batch = next(iter(dataloaders["train"][sess]))
print("batch type :", type(batch).__name__)
print("inputs     :", tuple(batch.inputs.shape), " (b, C=3, t, h, w)")
print("targets    :", tuple(batch.targets.shape), " (b, t, n)")
print("pupil_center:", tuple(batch.pupil_center.shape), " (b, 2, t)")

Finally the exact objects `train.py:69-75` hands to Lightning: a `LongCycler` (round-robins sessions each step, for training) wrapped in a `DataLoader(batch_size=None)`, and a `ShortCycler` (iterates each session once, for validation). Each yields `(session_id, QiuDataPoint)`.

In [ ]:
import torch.utils.data as tud
from openretina.data_io.cyclers import LongCycler, ShortCycler

train_loader = tud.DataLoader(
    LongCycler(dataloaders["train"], shuffle=True),
    batch_size=None, num_workers=0, pin_memory=False,
)
valid_loader = ShortCycler(dataloaders["validation"])

sid, dp = next(iter(train_loader))
print("one LongCycler step ->", sid)
print("   inputs", tuple(dp.inputs.shape), "| targets", tuple(dp.targets.shape), "| pupil", tuple(dp.pupil_center.shape))
# NB: ShortCycler has no len(); report the number of validation sessions instead.
print("train steps/epoch:", len(train_loader), "| validation sessions:", len(dataloaders["validation"]))

## 4. Build the model & run a forward pass

The qiu model config has no top-level `_target_`, so `train.py:104-112` takes the else-branch: it sets `n_neurons_dict` from the data and constructs `UnifiedCoreReadout(data_info=..., **cfg.model)` directly. `UnifiedCoreReadout.__init__` then resolves `core.channels` from `in_shape[0]` + `hidden_channels`, computes the readout `in_shape` from the core, and builds the shifter keyed by session (`core_readout.py:361-390`).

In [ ]:
from openretina.models.core_readout import UnifiedCoreReadout

cfg.model.n_neurons_dict = data_info["n_neurons_dict"]  # mirrors train.py:106
model = UnifiedCoreReadout(data_info=data_info, **cfg.model)

n_params = sum(p.numel() for p in model.parameters())
print(f"total params: {n_params:,}")
print("readout sessions:", list(model.readout.keys()))
print("shifter sessions:", list(model.shifter.keys()) if model.shifter is not None else None)
print("loss / eval     :", type(model.loss).__name__, "/", type(model.evaluation_loss).__name__)
print(model.core)

Forward pass on the batch from stage 3. The core's un-padded temporal convs crop `sum(kernel-1) = 18` frames, so `T_out = t - 18`. With `pupil_center` supplied the shifter is active; passing `pupil_center=None` reproduces the pre-shifter path exactly (the regression guarantee from decision D1).

In [ ]:
import torch
from einops import rearrange

model.eval()
with torch.no_grad():
    out = model(batch.inputs, data_key=sess, pupil_center=batch.pupil_center)
    out_no_shift = model(batch.inputs, data_key=sess, pupil_center=None)

t_in = batch.inputs.shape[2]
cut = t_in - out.shape[1]
print(f"input t={t_in}  ->  output {tuple(out.shape)}  (b, T_out, N);  T_out = t - {cut}")

# Prove the shifter is active: it maps the time-aligned pupil center to a per-sample readout-grid shift.
pupil_aligned = rearrange(batch.pupil_center[:, :, cut:], "b two t -> (b t) two")
shift = model.shifter(pupil_aligned, sess)
print(f"shifter output: shape {tuple(shift.shape)}  abs-max {shift.abs().max():.3f} grid units  (nonzero: {(shift != 0).any().item()})")
print(f"output delta (shift on vs off), abs-max: {(out - out_no_shift).abs().max():.2e}")
print("   (delta is small on an untrained model / a single clip; it grows once the core learns spatial structure)")

# The Poisson loss crops targets to the output length internally (lag = t - T_out).
loss = model.loss(out, batch.targets)
print(f"PoissonLoss3d on this batch: {loss.item():.3f}  (finite: {torch.isfinite(loss).item()})")

## 5. Run a (smoke) training loop

`train.py:131-134` builds a Lightning `Trainer` from `cfg.trainer` and calls `trainer.fit(model, train_loader, valid_loader)`. Here we build a **deliberately tiny** trainer instead — a couple of batches on CPU, no checkpointing/logging — just to confirm the wiring trains and the loss stays finite.

> **This is not the Definition of Done.** A real epoch over all 10 sessions needs a big-RAM / GPU box. On the cluster, run:
> ```bash
> uv run openretina train --config-name qiu_2026_core_readout trainer=debug
> ```
> On CPU each batch here can take seconds-to-minutes; keep `SMOKE_TRAIN_BATCHES` small.

In [ ]:
import lightning

lightning.pytorch.seed_everything(cfg.seed, verbose=False)

trainer = lightning.Trainer(
    max_epochs=1,
    limit_train_batches=SMOKE_TRAIN_BATCHES,
    limit_val_batches=SMOKE_VAL_BATCHES,
    accelerator=ACCELERATOR,
    devices=1,
    logger=False,
    enable_checkpointing=False,
    enable_progress_bar=True,
)
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=valid_loader)

print("\nlogged metrics:")
for k, v in trainer.callback_metrics.items():
    print(f"   {k}: {float(v):.4f}")

## Where to go from here

* **Scale up sessions/batch:** set `SESSIONS = None` (all 10) and `BATCH_SIZE = 32` in the config cell — only on a big-RAM box. Re-run from stage 1.
* **Full training + checkpoint (the DoD):** use the CLI on the cluster — `uv run openretina train --config-name qiu_2026_core_readout trainer=debug`. The notebook path and the CLI path build the *same* objects; this notebook just lets you watch each one.
* **Inspect a trained checkpoint:** load it and call `model.forward(stimulus, data_key=..., pupil_center=...)` for any session key.
* **Open threads:** CASCADE spike inference and the real-data test suite — see the *Remaining work* section of `qiu_2026_integration_plan.md`.